# -라이브러리 임포트

In [3]:
import subprocess
import os
import time
import itertools
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
print("done")

done


# -사설서버(공기계) 를 통해 축적한 데이터를 데이터프레임으로 전환하기 좋게 처리

In [6]:
loa_path = "lostarkData.json"

print(os.path.isdir("LostarkDataDownloader"))
print(os.path.isfile('LostarkDataDownloader\\SimpleTest.exe'))

if os.path.isfile(loa_path):
    os.remove(loa_path)

subprocess.run(["powershell", "Start-Process", "LostarkDataDownloader\\SimpleTest.exe", "-Wait"])
print("done")

True
True
done


# -데이터셋 전처리
1. 데이터셋의 맨 앞과 맨 뒤의 날짜기준 1분간격으로 새 칼럼 'date' 포함 df 생성
2. 해당 df에 데이터셋을 left join
3. 결측치는 ffill
4. 데이터 용량이 너무커서 타입전환 (float64 -> float32)

In [5]:
loa_path = "lostarkData.json"
temp_df = pd.read_json(loa_path)

one_min = pd.Timedelta(minutes=1)
head_time = temp_df.date[0]
tail_time = temp_df.date[len(temp_df.date) - 1]


df = []
while head_time <= tail_time:
  df.append(head_time)
  head_time += one_min
df = pd.DataFrame(df, columns=["date"])

df = df.merge(temp_df, left_on='date', right_on='date', how='left')
df = df.replace("nan", np.nan)
df = df.ffill()

for col in df.columns[1:]:
    df[col] = df[col].astype("float32")

print(df)
print("\n" * 5)
print(df.info())

                      date  일반 운명의 수호석  일반 운명의 파괴석  고급 운명의 파편 주머니(소)  \
0      2025-09-27 04:18:00         2.0        37.0             175.0   
1      2025-09-27 04:19:00         2.0        37.0             175.0   
2      2025-09-27 04:20:00         2.0        37.0             175.0   
3      2025-09-27 04:21:00         2.0        38.0             177.0   
4      2025-09-27 04:22:00         2.0        38.0             177.0   
...                    ...         ...         ...               ...   
100738 2025-12-06 01:58:00         2.0        85.0             311.0   
100739 2025-12-06 01:59:00         3.0        86.0             341.0   
100740 2025-12-06 02:00:00         3.0        86.0             341.0   
100741 2025-12-06 02:01:00         3.0        86.0             341.0   
100742 2025-12-06 02:02:00         3.0        80.0             341.0   

        희귀 운명의 돌파석  희귀 아비도스 융화 재료  희귀 운명의 파편 주머니(중)  영웅 빙하의 숨결  영웅 용암의 숨결  \
0              6.0           91.0             354.0      1

## 사전 지식
### 시계열 데이터를 학습시키기 위해선 과거의 데이터묶음을 인풋으로
### 데이터묶음의 마지막 시점의 바로 다음 시점의 데이터를 아웃풋으로
### 쌍을 이루어 만들어야함

# -모델 학습 전략 수립
### 그리드서치를 이용, 아래는 변화를 주는 파라미터
1. 데이터묶음 크기 (1\~14분전, 1\~30분전)
2. 퍼셉트론 개수 (64개, 128개)
3. 드롭아웃 비율 (10%, 20%)

In [8]:
OUT_DIR = "./small_grid_results"
os.makedirs(OUT_DIR, exist_ok=True)

FEATURE_COLS = [c for c in df.columns if c != "date"]
n_features = len(FEATURE_COLS)

# Small grid (8 combos)
grid = {
    "window_size": [14, 30],
    "lstm_units": [64, 128],
    "dropout": [0.1, 0.2],
}
# fixed params
batch_size = 64
learning_rate = 1e-3

# splits and training
TEST_RATIO = 0.15
VAL_RATIO = 0.15
EPOCHS = 50          # can increase for final runs
PATIENCE = 6

# -------------------------
# Utilities
# -------------------------
def create_sequences(values: np.ndarray, window_size: int, pred_horizon: int = 1):
    X, y = [], []
    T = len(values)
    for start in range(0, T - window_size - pred_horizon + 1):
        end = start + window_size
        X.append(values[start:end])
        y.append(values[end + pred_horizon - 1])
    return np.array(X), np.array(y)

def build_model(window_size, n_features, lstm_units, dropout, learning_rate):
    model = Sequential()
    model.add(LSTM(lstm_units, input_shape=(window_size, n_features)))
    if dropout and dropout > 0:
        model.add(Dropout(dropout))
    model.add(Dense(max(64, lstm_units // 2), activation="relu"))
    model.add(Dense(n_features, activation="linear"))
    opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=opt, loss="mse", metrics=["mae"])
    return model

# -------------------------
# Raw values (original scale)
# -------------------------
values = df[FEATURE_COLS].values.astype("float32")
T_total = len(values)
print("Total timesteps:", T_total, "n_features:", n_features)

# -------------------------
# Grid combos
# -------------------------
keys, vals = zip(*grid.items())
combinations = [dict(zip(keys, v)) for v in itertools.product(*vals)]
print("Total combos to run:", len(combinations))

results = []

for idx, combo in enumerate(combinations, 1):
    start_time = time.time()
    ws = combo["window_size"]
    lstm_units = combo["lstm_units"]
    dropout = combo["dropout"]

    combo_name = f"ws{ws}_u{lstm_units}_d{dropout}"
    print(f"\n[{idx}/{len(combinations)}] START combo: {combo_name}")

    # 1) Create sequences on raw values to get sample counts for split sizing
    X_all_raw, y_all_raw = create_sequences(values, ws, pred_horizon=1)
    n_samples = len(X_all_raw)
    n_test = int(n_samples * TEST_RATIO)
    n_val = int(n_samples * VAL_RATIO)
    n_train = n_samples - n_val - n_test
    if n_train <= 0:
        print("  -> window too large for dataset, skipping")
        continue

    # 2) Fit scaler on raw rows that correspond to the training region.
    # training sequences use raw rows indices [0 .. n_train+ws-1]
    train_raw_end = n_train + ws - 1
    scaler = MinMaxScaler()
    scaler.fit(values[: train_raw_end + 1])  # only train rows

    # 3) Scale full raw values and recreate sequences (so X,y in scaled space)
    values_scaled = scaler.transform(values)
    X_all, y_all = create_sequences(values_scaled, ws, pred_horizon=1)

    # 4) split sequences timewise
    X_train = X_all[:n_train]
    Y_train = y_all[:n_train]
    X_val = X_all[n_train:n_train + n_val]
    Y_val = y_all[n_train:n_train + n_val]
    X_test = X_all[n_train + n_val:]
    Y_test = y_all[n_train + n_val:]

    print(f"  samples -> train: {X_train.shape[0]}, val: {X_val.shape[0]}, test: {X_test.shape[0]}")

    # 5) build model
    tf.keras.backend.clear_session()
    model = build_model(ws, n_features, lstm_units, dropout, learning_rate)

    # callbacks
    model_path = os.path.join(OUT_DIR, f"best_{combo_name}.h5")
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
        ModelCheckpoint(model_path, monitor="val_loss", save_best_only=True, verbose=0)
    ]

    # 6) train
    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=EPOCHS,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=2
    )

    # 7) evaluate: predict on test and inverse transform
    pred_scaled = model.predict(X_test)
    true_scaled = Y_test
    pred = scaler.inverse_transform(pred_scaled)
    true = scaler.inverse_transform(true_scaled)

    # metrics per feature -> then mean
    rmse_per_feature = np.sqrt(np.mean((pred - true) ** 2, axis=0))
    mae_per_feature = np.mean(np.abs(pred - true), axis=0)
    rmse_mean = float(np.mean(rmse_per_feature))
    mae_mean = float(np.mean(mae_per_feature))
    val_loss_min = float(min(history.history["val_loss"])) if "val_loss" in history.history else None
    elapsed = time.time() - start_time

    # 8) save artifacts
    joblib.dump(scaler, os.path.join(OUT_DIR, f"scaler_{combo_name}.pkl"))
    # model saved by ModelCheckpoint; save final as well
    model.save(os.path.join(OUT_DIR, f"final_{combo_name}.h5"))
    # history
    pd.DataFrame(history.history).to_csv(os.path.join(OUT_DIR, f"history_{combo_name}.csv"), index=False)
    # small sample preds (first 5 features)
    sample_df = pd.DataFrame({
        "date": (df['date'].iloc[-len(true):].reset_index(drop=True) if "date" in df.columns else range(len(true)))
    })
    for fi in range(min(5, n_features)):
        sample_df[f"true_{FEATURE_COLS[fi]}"] = true[:, fi]
        sample_df[f"pred_{FEATURE_COLS[fi]}"] = pred[:, fi]
    sample_df.to_csv(os.path.join(OUT_DIR, f"sample_preds_{combo_name}.csv"), index=False)

    # 9) record result
    results.append({
        "combo_name": combo_name,
        "window_size": ws,
        "lstm_units": lstm_units,
        "dropout": dropout,
        "batch_size": batch_size,
        "learning_rate": learning_rate,
        "val_loss_min": val_loss_min,
        "test_rmse": rmse_mean,
        "test_mae": mae_mean,
        "train_samples": X_train.shape[0],
        "val_samples": X_val.shape[0],
        "test_samples": X_test.shape[0],
        "elapsed_sec": elapsed
    })

    print(f"  DONE {combo_name} | val_loss_min={val_loss_min:.6f} test_rmse={rmse_mean:.6f} elapsed={elapsed:.1f}s")

# Save summary
results_df = pd.DataFrame(results).sort_values("val_loss_min").reset_index(drop=True)
results_df.to_csv(os.path.join(OUT_DIR, "grid_search_results_summary.csv"), index=False)
print("\nGrid search finished. Summary:")
print(results_df.head())

Total timesteps: 100743 n_features: 92
Total combos to run: 8

[1/8] START combo: ws14_u64_d0.1
  samples -> train: 70511, val: 15109, test: 15109



C:\Users\jksjk\anaconda3\envs\inha2025\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/50


1102/1102 - 8s - 7ms/step - loss: 0.0081 - mae: 0.0520 - val_loss: 0.0032 - val_mae: 0.0373 - learning_rate: 0.0010
Epoch 2/50


1102/1102 - 5s - 5ms/step - loss: 0.0019 - mae: 0.0289 - val_loss: 0.0021 - val_mae: 0.0304 - learning_rate: 0.0010
Epoch 3/50


...

1102/1102 - 22s - 20ms/step - loss: 8.2469e-04 - mae: 0.0188 - val_loss: 9.5562e-04 - val_mae: 0.0189 - learning_rate: 5.0000e-04
Epoch 13/50

Epoch 13: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
1102/1102 - 17s - 16ms/step - loss: 8.1656e-04 - mae: 0.0187 - val_loss: 0.0010 - val_mae: 0.0197 - learning_rate: 5.0000e-04
Epoch 14/50


1102/1102 - 18s - 16ms/step - loss: 7.8986e-04 - mae: 0.0183 - val_loss: 9.5074e-04 - val_mae: 0.0188 - learning_rate: 2.5000e-04
Epoch 15/50
1102/1102 - 18s - 16ms/step - loss: 7.8559e-04 - mae: 0.0182 - val_loss: 9.5932e-04 - val_mae: 0.0189 - learning_rate: 2.5000e-04
Epoch 16/50

Epoch 16: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
1102/1102 - 17s - 16ms/step - loss: 7.7975e-04 - mae: 0.0181 - val_loss: 9.8563e-04 - val_mae: 0.0193 - learning_rate: 2.5000e-04
Epoch 17/50
1102/1102 - 18s - 16ms/step - loss: 7.6592e-04 - mae: 0.0179 - val_loss: 9.5348e-04 - val_mae: 0.0189 - learning_rate: 1.2500e-04
Epoch 18/50
1102/1102 - 18s - 16ms/step - loss: 7.6381e-04 - mae: 0.0179 - val_loss: 9.5949e-04 - val_mae: 0.0191 - learning_rate: 1.2500e-04
Epoch 19/50

Epoch 19: ReduceLROnPlateau reducing learning rate to 6.25000029685907e-05.


1102/1102 - 18s - 16ms/step - loss: 7.6157e-04 - mae: 0.0178 - val_loss: 9.4383e-04 - val_mae: 0.0187 - learning_rate: 1.2500e-04
Epoch 20/50
1102/1102 - 19s - 17ms/step - loss: 7.5354e-04 - mae: 0.0177 - val_loss: 9.5338e-04 - val_mae: 0.0188 - learning_rate: 6.2500e-05
Epoch 21/50
1102/1102 - 19s - 17ms/step - loss: 7.5112e-04 - mae: 0.0177 - val_loss: 9.4707e-04 - val_mae: 0.0187 - learning_rate: 6.2500e-05
Epoch 22/50


1102/1102 - 19s - 18ms/step - loss: 7.5149e-04 - mae: 0.0177 - val_loss: 9.3879e-04 - val_mae: 0.0186 - learning_rate: 6.2500e-05
Epoch 23/50
1102/1102 - 18s - 17ms/step - loss: 7.4721e-04 - mae: 0.0176 - val_loss: 9.3922e-04 - val_mae: 0.0186 - learning_rate: 6.2500e-05
Epoch 24/50
1102/1102 - 18s - 17ms/step - loss: 7.4824e-04 - mae: 0.0176 - val_loss: 9.4654e-04 - val_mae: 0.0186 - learning_rate: 6.2500e-05
Epoch 25/50

Epoch 25: ReduceLROnPlateau reducing learning rate to 3.125000148429535e-05.
1102/1102 - 18s - 17ms/step - loss: 7.4575e-04 - mae: 0.0176 - val_loss: 9.5482e-04 - val_mae: 0.0188 - learning_rate: 6.2500e-05
Epoch 26/50
1102/1102 - 18s - 16ms/step - loss: 7.4181e-04 - mae: 0.0176 - val_loss: 9.4947e-04 - val_mae: 0.0187 - learning_rate: 3.1250e-05
Epoch 27/50
1102/1102 - 18s - 16ms/step - loss: 7.4039e-04 - mae: 0.0175 - val_loss: 9.5249e-04 - val_mae: 0.0188 - learning_rate: 3.1250e-05
Epoch 28/50

Epoch 28: ReduceLROnPlateau reducing learning rate to 1.5625000742147

  DONE ws30_u128_d0.2 | val_loss_min=0.000939 test_rmse=1328.254272 elapsed=518.2s

Grid search finished. Summary:
       combo_name  window_size  lstm_units  dropout  batch_size  \
0  ws14_u128_d0.1           14         128      0.1          64   
1  ws30_u128_d0.1           30         128      0.1          64   
2   ws30_u64_d0.1           30          64      0.1          64   
3  ws14_u128_d0.2           14         128      0.2          64   
4   ws14_u64_d0.1           14          64      0.1          64   

   learning_rate  val_loss_min    test_rmse     test_mae  train_samples  \
0          0.001      0.000630  1122.791870   895.529358          70511   
1          0.001      0.000637   915.095886   714.554871          70501   
2          0.001      0.000843  1250.547974   994.820251          70501   
3          0.001      0.000852  1274.019043  1016.094910          70511   
4          0.001      0.000936  1427.211670  1156.726440          70511   

   val_samples  test_samples  e

# -> 결과
### 데이터묶음30개, 퍼셉트론 128개, 드롭아웃 비율 10%
### 일 때 테스트데이터를 통한 오차가 제일낮음

# -모델 학습 전략 수립 2
### 그리드서치를 이용, 아래는 변화를 주는 파라미터
1. 데이터묶음 크기 (1\~14분전, 1\~30분전, 1\~45분전, 1\~60분전)

In [9]:
OUT_DIR = "./window_grid_results"
os.makedirs(OUT_DIR, exist_ok=True)

FEATURE_COLS = [c for c in df.columns if c != "date"]
n_features = len(FEATURE_COLS)

# WINDOWS to try (add/remove sizes if you want)
WINDOW_SIZES = [14, 30, 45, 60]

# fixed hyperparams (from best combo earlier)
LSTM_UNITS = 128
DROPOUT = 0.1
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

# splits and training
TEST_RATIO = 0.15
VAL_RATIO = 0.15
EPOCHS = 80        # 필요시 줄이거나 늘리세요
PATIENCE = 8

# -------------------------
# Utilities
# -------------------------
def create_sequences(values: np.ndarray, window_size: int, pred_horizon: int = 1):
    X, y = [], []
    T = len(values)
    for start in range(0, T - window_size - pred_horizon + 1):
        end = start + window_size
        X.append(values[start:end])
        y.append(values[end + pred_horizon - 1])
    return np.array(X), np.array(y)

def build_model(window_size, n_features, lstm_units, dropout, learning_rate):
    model = Sequential()
    model.add(LSTM(lstm_units, input_shape=(window_size, n_features)))
    if dropout and dropout > 0:
        model.add(Dropout(dropout))
    model.add(Dense(max(64, lstm_units // 2), activation="relu"))
    model.add(Dense(n_features, activation="linear"))
    opt = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=opt, loss="mse", metrics=["mae"])
    return model

# -------------------------
# raw values
# -------------------------
values = df[FEATURE_COLS].values.astype("float32")
T_total = len(values)
print("Total timesteps:", T_total, "n_features:", n_features)

results = []

# loop windows
for ws in WINDOW_SIZES:
    start_time = time.time()
    combo_name = f"ws{ws}_u{LSTM_UNITS}_d{DROPOUT}"
    print(f"\n=== START: {combo_name} ===")

    # 1) create sequences on raw to get counts
    X_all_raw, y_all_raw = create_sequences(values, ws, pred_horizon=1)
    n_samples = len(X_all_raw)
    n_test = int(n_samples * TEST_RATIO)
    n_val = int(n_samples * VAL_RATIO)
    n_train = n_samples - n_val - n_test

    print("samples (total/train/val/test):", n_samples, n_train, n_val, n_test)
    if n_train <= 0:
        print("  -> window_size too large for dataset, skip.")
        continue

    # 2) fit scaler on raw training rows
    train_raw_end = n_train + ws - 1
    scaler = MinMaxScaler()
    scaler.fit(values[: train_raw_end + 1])

    # 3) scale full values and recreate sequences
    values_scaled = scaler.transform(values)
    X_all, y_all = create_sequences(values_scaled, ws, pred_horizon=1)

    # 4) split timewise
    X_train = X_all[:n_train]
    Y_train = y_all[:n_train]
    X_val = X_all[n_train:n_train + n_val]
    Y_val = y_all[n_train:n_train + n_val]
    X_test = X_all[n_train + n_val:]
    Y_test = y_all[n_train + n_val:]

    print("Shapes ->", X_train.shape, X_val.shape, X_test.shape)

    # 5) build model
    tf.keras.backend.clear_session()
    model = build_model(ws, n_features, LSTM_UNITS, DROPOUT, LEARNING_RATE)

    # callbacks
    model_path = os.path.join(OUT_DIR, f"best_{combo_name}.h5")
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, verbose=1),
        ModelCheckpoint(model_path, monitor="val_loss", save_best_only=True, verbose=0)
    ]

    # 6) train
    history = model.fit(
        X_train, Y_train,
        validation_data=(X_val, Y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        verbose=2
    )

    # 7) evaluate
    pred_scaled = model.predict(X_test)
    true_scaled = Y_test
    pred = scaler.inverse_transform(pred_scaled)
    true = scaler.inverse_transform(true_scaled)

    rmse_per_feature = np.sqrt(np.mean((pred - true) ** 2, axis=0))
    mae_per_feature = np.mean(np.abs(pred - true), axis=0)
    rmse_mean = float(np.mean(rmse_per_feature))
    mae_mean = float(np.mean(mae_per_feature))
    val_loss_min = float(min(history.history["val_loss"])) if "val_loss" in history.history else None
    elapsed = time.time() - start_time

    # 8) save artifacts
    joblib.dump(scaler, os.path.join(OUT_DIR, f"scaler_{combo_name}.pkl"))
    model.save(os.path.join(OUT_DIR, f"final_{combo_name}.h5"))
    pd.DataFrame(history.history).to_csv(os.path.join(OUT_DIR, f"history_{combo_name}.csv"), index=False)

    # sample preds (first 5 features)
    sample_df = pd.DataFrame({
        "date": (df['date'].iloc[-len(true):].reset_index(drop=True) if "date" in df.columns else range(len(true)))
    })
    for fi in range(min(5, n_features)):
        sample_df[f"true_{FEATURE_COLS[fi]}"] = true[:, fi]
        sample_df[f"pred_{FEATURE_COLS[fi]}"] = pred[:, fi]
    sample_df.to_csv(os.path.join(OUT_DIR, f"sample_preds_{combo_name}.csv"), index=False)

    # record
    results.append({
        "combo_name": combo_name,
        "window_size": ws,
        "lstm_units": LSTM_UNITS,
        "dropout": DROPOUT,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "val_loss_min": val_loss_min,
        "test_rmse": rmse_mean,
        "test_mae": mae_mean,
        "train_samples": X_train.shape[0],
        "val_samples": X_val.shape[0],
        "test_samples": X_test.shape[0],
        "elapsed_sec": elapsed
    })

    print(f"=== DONE {combo_name} | val_loss_min={val_loss_min:.6f} test_rmse={rmse_mean:.3f} elapsed={elapsed:.1f}s ===")

# Save summary
results_df = pd.DataFrame(results).sort_values("val_loss_min").reset_index(drop=True)
results_df.to_csv(os.path.join(OUT_DIR, "window_grid_results_summary.csv"), index=False)
print("\nWindow-grid finished. Summary:")
print(results_df)

Total timesteps: 100743 n_features: 92

=== START: ws14_u128_d0.1 ===
samples (total/train/val/test): 100729 70511 15109 15109
Shapes -> (70511, 14, 92) (15109, 14, 92) (15109, 14, 92)
Epoch 1/80


C:\Users\jksjk\anaconda3\envs\inha2025\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


1102/1102 - 11s - 10ms/step - loss: 0.0079 - mae: 0.0506 - val_loss: 0.0028 - val_mae: 0.0347 - learning_rate: 0.0010
Epoch 2/80


1102/1102 - 9s - 8ms/step - loss: 0.0017 - mae: 0.0282 - val_loss: 0.0020 - val_mae: 0.0297 - learning_rate: 0.0010
Epoch 3/80


1102/1102 - 10s - 9ms/step - loss: 0.0012 - mae: 0.0233 - val_loss: 0.0016 - val_mae: 0.0262 - learning_rate: 0.0010
Epoch 4/80


...

1102/1102 - 44s - 40ms/step - loss: 6.1417e-04 - mae: 0.0159 - val_loss: 7.8606e-04 - val_mae: 0.0169 - learning_rate: 0.0010
Epoch 17/80

Epoch 17: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
1102/1102 - 43s - 39ms/step - loss: 6.0769e-04 - mae: 0.0157 - val_loss: 8.0748e-04 - val_mae: 0.0168 - learning_rate: 0.0010
Epoch 18/80


1102/1102 - 44s - 40ms/step - loss: 5.7488e-04 - mae: 0.0151 - val_loss: 7.7818e-04 - val_mae: 0.0162 - learning_rate: 5.0000e-04
Epoch 19/80


1102/1102 - 44s - 40ms/step - loss: 5.7356e-04 - mae: 0.0151 - val_loss: 7.7330e-04 - val_mae: 0.0163 - learning_rate: 5.0000e-04
Epoch 20/80


1102/1102 - 43s - 39ms/step - loss: 5.7131e-04 - mae: 0.0150 - val_loss: 7.2559e-04 - val_mae: 0.0154 - learning_rate: 5.0000e-04
Epoch 21/80
1102/1102 - 43s - 39ms/step - loss: 5.6843e-04 - mae: 0.0150 - val_loss: 7.8406e-04 - val_mae: 0.0163 - learning_rate: 5.0000e-04
Epoch 22/80
1102/1102 - 44s - 40ms/step - loss: 5.6478e-04 - mae: 0.0149 - val_loss: 7.5621e-04 - val_mae: 0.0161 - learning_rate: 5.0000e-04
Epoch 23/80
1102/1102 - 44s - 40ms/step - loss: 5.6331e-04 - mae: 0.0148 - val_loss: 7.5825e-04 - val_mae: 0.0160 - learning_rate: 5.0000e-04
Epoch 24/80

Epoch 24: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.
1102/1102 - 43s - 39ms/step - loss: 5.6112e-04 - mae: 0.0148 - val_loss: 7.6717e-04 - val_mae: 0.0158 - learning_rate: 5.0000e-04
Epoch 25/80
1102/1102 - 43s - 39ms/step - loss: 5.4567e-04 - mae: 0.0145 - val_loss: 7.4783e-04 - val_mae: 0.0154 - learning_rate: 2.5000e-04
Epoch 26/80
1102/1102 - 43s - 39ms/step - loss: 5.4458e-04 - mae: 0.0145 - val_los

=== DONE ws60_u128_d0.1 | val_loss_min=0.000726 test_rmse=1086.053 elapsed=1297.9s ===

Window-grid finished. Summary:
       combo_name  window_size  lstm_units  dropout  batch_size  \
0  ws14_u128_d0.1           14         128      0.1          64   
1  ws60_u128_d0.1           60         128      0.1          64   
2  ws45_u128_d0.1           45         128      0.1          64   
3  ws30_u128_d0.1           30         128      0.1          64   

   learning_rate  val_loss_min    test_rmse    test_mae  train_samples  \
0          0.001      0.000679  1101.987305  876.159363          70511   
1          0.001      0.000726  1086.053223  868.317444          70479   
2          0.001      0.000747  1082.980591  873.162964          70490   
3          0.001      0.000789  1077.100342  862.179077          70501   

   val_samples  test_samples  elapsed_sec  
0        15109         15109   559.665469  
1        15102         15102  1297.890395  
2        15104         15104  1539.231730 

# -> 결과
### 데이터묶음이 많을수록 학습률은 좋으나
### 묶음이 30개일때 테스트오차가 제일낮음 (위의 경우 과적합 의심)

# -모델 학습
### 파라미터
1. 데이터묶음 30개
2. 검증데이터비율 15%
3. 테스트데이터비율 15%
4. 배치사이즈 128

그 외 학습전략 -> 조기종료, 데이터 0~1 정규화

In [6]:
WINDOW_SIZE = 30 # 과거 30일을 보고 다음 1일을 예측
PRED_HORIZON = 1 # 예측 horizon (여기선 1)
TEST_RATIO = 0.15
VAL_RATIO = 0.15
BATCH_SIZE = 128
EPOCHS = 100

def create_sequences(values: np.ndarray, window_size: int, pred_horizon: int = 1):
    X, y = [], []
    T = len(values)
    for start in range(0, T - window_size - pred_horizon + 1):
        end = start + window_size
        X.append(values[start:end])
        y.append(values[end + pred_horizon - 1])
    return np.array(X), np.array(y)

FEATURE_COLS = [c for c in df.columns if c != 'date']
values = df[FEATURE_COLS].values.astype('float32')

scaler = MinMaxScaler()
values_scaled = scaler.fit_transform(values)


X, y = create_sequences(values_scaled, WINDOW_SIZE, PRED_HORIZON)
print('X.shape, y.shape =', X.shape, y.shape)



n_total = len(X)
n_test = int(n_total * TEST_RATIO)
n_val = int(n_total * VAL_RATIO)
n_train = n_total - n_val - n_test

X_train = X[:n_train]
Y_train = y[:n_train]
X_val = X[n_train:n_train + n_val]
Y_val = y[n_train:n_train + n_val]
X_test = X[n_train + n_val:]
Y_test = y[n_train + n_val:]

print('Train/Val/Test shapes:', X_train.shape, X_val.shape, X_test.shape)


n_features = X_train.shape[2]
model = Sequential()
model.add(LSTM(256, input_shape=(WINDOW_SIZE, n_features), return_sequences=False))
model.add(Dropout(0.2))
model.add(Dense(128, activation='relu'))
model.add(Dense(n_features, activation='linear')) # 멀티아웃풋: 모든 피처를 예측


model.compile(optimizer='adam', loss='mse', metrics=['mae'])
model.summary()


callbacks = [
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4, verbose=1)
]

history = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=2
)


pred_scaled = model.predict(X_test)




pred = scaler.inverse_transform(pred_scaled)
true = scaler.inverse_transform(Y_test)



rmse_per_feature = np.sqrt(np.mean((pred - true) ** 2, axis=0))
mae_per_feature = np.mean(np.abs(pred - true), axis=0)


print('전체 피처 평균 RMSE:', np.mean(rmse_per_feature))
print('전체 피처 평균 MAE:', np.mean(mae_per_feature))


# 주요 피처(상위 몇개) 출력 예시
for i, col in enumerate(FEATURE_COLS[:10]):
    print(f'{col}: RMSE={rmse_per_feature[i]:.4f}, MAE={mae_per_feature[i]:.4f}')


X.shape, y.shape = (100713, 30, 92) (100713, 92)
Train/Val/Test shapes: (70501, 30, 92) (15106, 30, 92) (15106, 30, 92)


C:\Users\jksjk\anaconda3\envs\inha2025\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 256)                 │         357,376 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 92)                  │          11,868 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 402,140 (1.53 MB)

 Trainable params: 402,140 (1.53 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
551/551 - 30s - 55ms/step - loss: 0.0061 - mae: 0.0480 - val_loss: 0.0022 - val_mae: 0.0324 - learning_rate: 0.0010
Epoch 2/100
551/551 - 28s - 51ms/step - loss: 0.0015 - mae: 0.0277 - val_loss: 0.0015 - val_mae: 0.0260 - learning_rate: 0.0010
Epoch 3/100
551/551 - 28s - 52ms/step - loss: 0.0010 - mae: 0.0225 - val_loss: 0.0011 - val_mae: 0.0229 - learning_rate: 0.0010
Epoch 4/100
551/551 - 28s - 50ms/step - loss: 7.6585e-04 - mae: 0.0192 - val_loss: 8.7842e-04 - val_mae: 0.0199 - learning_rate: 0.0010
Epoch 5/100
551/551 - 27s - 50ms/step - loss: 6.5052e-04 - mae: 0.0176 - val_loss: 7.3268e-04 - val_mae: 0.0179 - learning_rate: 0.0010
Epoch 6/100
551/551 - 27s - 50ms/step - loss: 5.8513e-04 - mae: 0.0165 - val_loss: 6.5584e-04 - val_mae: 0.0165 - learning_rate: 0.0010
Epoch 7/100
551/551 - 28s - 51ms/step - loss: 5.4366e-04 - mae: 0.0158 - val_loss: 5.8959e-04 - val_mae: 0.0155 - learning_rate: 0.0010
Epoch 8/100
551/551 - 29s - 52ms/step - loss: 5.1564e-04 - mae: 0.0154 -

# -> 학습결과
### 전체적으로 준수하게 잘 됐음
### 시각화 그래프를 확인
깃허브 업로드 관련 용량이슈로 별첨함

In [11]:
model.save("loa_lstm.keras")

In [10]:
import plotly.graph_objects as go

# ====== 전제 변수 (이름이 다르면 변경) ======
# FEATURE_COLS : 리스트(92개) - feature 이름들
# true : numpy array, shape (n_test, n_features)
# pred : numpy array, shape (n_test, n_features)
# df : 원본 dataframe (date 컬럼이 있으면 x 축으로 사용)

# ====== x축 시간값 준비 ======
try:
    # test 구간의 날짜를 df에서 가져오는 시도
    # df['date'] 전체 길이에서 true 길이만큼 슬라이스
    x_vals = df['date'].iloc[-len(true):].tolist()
except Exception:
    x_vals = list(range(len(true)))

n_features = len(FEATURE_COLS)
traces = []

# 각 feature에 대해 Actual / Predicted trace 생성
for i, fname in enumerate(FEATURE_COLS):
    # Actual
    traces.append(go.Scatter(
        x=x_vals,
        y=true[:, i],
        mode='lines',
        name=f'{fname} - 실제값',
        visible=(i == 0)  # 처음에는 첫 피처만 보이도록
    ))
    # Predicted
    traces.append(go.Scatter(
        x=x_vals,
        y=pred[:, i],
        mode='lines',
        name=f'{fname} - 예측값',
        visible=(i == 0)
    ))

# ====== 드롭다운 버튼 생성 ======
# 각 버튼은 visibility 배열(2*n_features 길이)을 만들고, 해당 피처의 두 trace만 True로 설정
buttons = []
for i, fname in enumerate(FEATURE_COLS):
    vis = [False] * (2 * n_features)
    vis[2*i] = True      # Actual
    vis[2*i + 1] = True  # Predicted
    buttons.append(dict(
        label=fname,
        method="update",
        args=[{"visible": vis},
              {"title": f""}]
    ))

# 옵션: 'All' 버튼 (모두 켜기) — 성능 이슈 있을 수 있으니 필요하면 사용
buttons.insert(0, dict(
    label="전부 보기",
    method="update",
    args=[{"visible": [True] * (2 * n_features)},
          {"title": "Actual vs Predicted — All (overlay)"}]
))

# ====== Figure 생성 ======
fig = go.Figure(data=traces)

fig.update_layout(
    updatemenus=[dict(
        active=1,
        buttons=buttons,
        x=0.0,
        y=1.15,
        xanchor='left',
        yanchor='top',
        direction='down'
    )],
    title=f"",
    xaxis_title="Time",
    yaxis_title="Value",
    showlegend=True,
    height=600,
    template='plotly_white'
)

print("done")

done


In [29]:
fig.write_html("로스트아크 아이템 시세예측.html")
print("done")

done
